In [1]:
import sys
import numpy as np
import yaml
from datetime import datetime


from pathlib import Path
from loguru import logger

Setup functions

In [45]:
def setup_logger():
    logger.remove()
    logger.add(
        sys.stderr,
        format="<green>{time:YYYY-MM-DD HH:mm:ss}</green> | <level>{level: <8}</level> | <level>{message}</level>",
        level="INFO",
        colorize=True
    )

def load_config(config_path):
    if Path(config_path).exists():
        with open(config_path, 'r') as f:
            config = yaml.safe_load(f)        
        logger.success(f"Loaded configuration from: {config_path}")
        return config
    else:
        raise FileNotFoundError(f"Configuration file not found: {config_path}")

def load_events(events_path, cont_path):
    event_ts  = np.load(events_path, mmap_mode='r')
    cont_ts   = np.load(cont_path, mmap_mode='r')
    states    = np.load(events_path.replace('timestamps.npy', 'states.npy'), mmap_mode='r')
    return event_ts, cont_ts, states


def validate_config(config):
    for k in ['recording_paths', 'remote_output', 'local_output', 'save_kwargs']:
        if k not in config:
            raise ValueError(f"Missing required configuration parameter: {k}")
    
    config.setdefault('session_name',   'default_session')
    config.setdefault('probe_filter',   None)
    config.setdefault('target_fs',      None)

    if not config['recording_paths']:
        raise ValueError("No recording paths specified in configuration.")
    
    # These keywords are used for stream/probe identification
    reserved_keywords = ['ADC', 'Adc', 'ProbeA', 'ProbeB', 'ProbeC', 'ProbeD']
    
    # Validate recording paths
    path_objs = []
    for path in config['recording_paths']:
        if not Path(path).exists():
            raise FileNotFoundError(f"Recording path not found: {path}")
        
        # Check parent folders for reserved keywords
        path_obj = Path(path).resolve()
        parent_parts = path_obj.parts
        
        for keyword in reserved_keywords:
            for part in parent_parts:
                if keyword in part:
                    raise ValueError(
                        f"Recording path contains reserved keyword '{keyword}' in parent folder: {path}\n")
        path_objs.append(path_obj)
    config['recording_paths'] = path_objs

    if not Path(config['local_output']).exists():
        raise FileNotFoundError(f"Local output directory not found: {config['local_output']}")

    if not Path(config['remote_output']).exists():
        raise FileNotFoundError(f"Remote output directory not found: {config['remote_output']}")

    return config

def setup(config_path='config.yaml'):
    # Load and validate config
    setup_logger()
    protocol = validate_config(load_config(config_path))
    
    # Extract config parameters
    session_name = protocol['session_name']
    session_name = session_name.strip().replace(' ', '_')
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    protocol['session_name'] = session_name
    
    # Setup output paths
    remote_output = Path(protocol['remote_output']).resolve()
    local_output = Path(protocol['local_output']).resolve()

    protocol['remote_output'] = remote_output / session_name
    protocol['local_output'] = local_output / session_name

    # Setup logging file in session folder root
    protocol['local_output'].mkdir(parents=True, exist_ok=True)
    log_path = protocol['local_output'] / f'{session_name}_{timestamp}.log'

    logger.add(
        log_path,
        rotation="500 MB",
        retention="10 days",
        level="DEBUG",
        format="{time:YYYY-MM-DD HH:mm:ss} | {level: <8} | {message}"
    )
    logger.success(f"Logger configured at {log_path}")

    logger.info(f"Pipeline configuration")
    logger.info(f"Session: {session_name}")
    logger.info(f"Recording sessions: {len(protocol['recording_paths'])}")
    logger.info(f"Local output: {protocol['local_output']}")
    logger.info(f"Remote output: {protocol['remote_output']}")
    logger.debug(f"  Parallel jobs: {protocol['save_kwargs']['n_jobs']}")
    logger.debug(f"  EEG downsampling frequency: {protocol['target_fs']}")
    return protocol

In [46]:
p = setup(r"R:\Basic_Sciences\Phys\SenzaiLab\kilosort_output\configs\config.yaml")
p

2025-11-04 12:42:02 | SUCCESS  | Loaded configuration from: R:\Basic_Sciences\Phys\SenzaiLab\kilosort_output\configs\config.yaml


2025-11-04 12:42:02 | SUCCESS  | Logger configured at E:\kilosort_output\AA001_Day2\AA001_Day2_20251104_124202.log
2025-11-04 12:42:02 | INFO     | Pipeline configuration
2025-11-04 12:42:02 | INFO     | Session: AA001_Day2
2025-11-04 12:42:02 | INFO     | Recording sessions: 4
2025-11-04 12:42:02 | INFO     | Local output: E:\kilosort_output\AA001_Day2
2025-11-04 12:42:02 | INFO     | Remote output: \\fsmresfiles.fsm.northwestern.edu\FSMResfiles\Basic_Sciences\Phys\SenzaiLab\kilosort_output\AA001_Day2


{'session_name': 'AA001_Day2',
 'recording_paths': [WindowsPath('//fsmresfiles.fsm.northwestern.edu/FSMResfiles/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/VisualStimuli/AA001_2025-10-09_12-31-37_4probe_RSC_ADn_Visual'),
  WindowsPath('//fsmresfiles.fsm.northwestern.edu/FSMResfiles/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/OpenField_Homecage/AA001_2025-10-09_13-41-27_4Probe_RSC_ADn_RecOpenField'),
  WindowsPath('//fsmresfiles.fsm.northwestern.edu/FSMResfiles/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/OpenField_Homecage/AA001_2025-10-09_14-22-26_4Probe_RSC_ADn_RecOpenField'),
  WindowsPath('//fsmresfiles.fsm.northwestern.edu/FSMResfiles/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/OpenField_Homecage/AA001_2025-10-09_14-57-57_4Probe_RSC_ADn_RecOpenField')],
 'local_output': WindowsPath('E:/kilosort_output/AA001_Day2'),
 'remote_output': WindowsPath('//fsmresfiles.fsm.northwestern.edu/FSMResfiles/Basic_Sciences/Phys/SenzaiLab/kilosort_output/AA001_Day2'),
 'fs': 30000.0,
 'target_fs

In [47]:
import spikeinterface.extractors as se
import spikeinterface as si

p['probe_filter'] = ['OneBox-ADC']  # Always include ADC
p['probe_filter'].extend(['ProbeA', 'ProbeB'])

In [35]:
test_paths = ['R:\\Basic_Sciences\\Phys\\SenzaiLab\\Tuguldur\\test\\AA001_2025-10-16_14-01-10_1probe_test',
              r"R:\Basic_Sciences\Phys\SenzaiLab\Tuguldur\test\005_2025-10-16_16-42-37_1probe_test"]

In [48]:
# PARSE
import re

def parse_timestamps(rec_paths, probe_filter):
    """Recursively parse timestamps.npy files from recording paths."""
    timestamps = {probe: {'event': [], 'cont': []} for probe in probe_filter}

    for session_idx, session_path in enumerate(rec_paths, 1):
        logger.debug(f"Session {session_idx}/{len(rec_paths)}: {session_path.name}") 
        # Recursively get all timestamps.npy files
        ts_files = list(session_path.glob('**/timestamps.npy'))
        # Sort by recording number extracted from filename
        ts_files = sorted(ts_files, key=lambda p: int(re.search(r'recording(\d+)', str(p)).group(1)))
        for ts_file in ts_files:
            ts_file = str(ts_file)
            for probe in probe_filter:
                if probe in ts_file:
                    if 'events' in ts_file:
                        timestamps[probe]['event'].append(ts_file)
                    elif 'continuous' in ts_file:
                        timestamps[probe]['cont'].append(ts_file)
    return timestamps

In [ ]:
def log_recording(rec, name="Recording"):
    """Log recordings."""
    n_channels  = rec.get_num_channels()
    duration    = rec.get_total_duration()
    fs          = rec.get_sampling_frequency()
    file_size   = rec.get_total_memory_size()
    dtype       = rec.get_dtype()
    logger.info(f"{name}:{n_channels} ch, {duration:.1f}s @ {fs/1000:.1f} kHz {dtype} ({format_file_size(file_size)})")
    # Log filepath if available
    if hasattr(rec, '_kwargs') and 'folder_path' in rec._kwargs:
        filepath = rec._kwargs['folder_path']
        logger.debug(f"Filepath: {filepath}")

def downsample_eeg(eeg_folder, rec, target_fs, chunk_duration=60):
    """Downsample recording to target_fs and save as binary."""
    eeg_file = eeg_folder / 'eeg_data.dat'
    if eeg_file.exists():
        logger.info(f"EEG data already exists, skipping downsampling")
        return
    
    logger.info(f"Downsampling EEG to: {eeg_folder}")
    fs = rec.get_sampling_frequency()
    decimation_factor = int(fs / target_fs)
    
    logger.info(f"Decimation factor: {decimation_factor}")
    logger.info(f"Output rate: {fs/decimation_factor:.2f} Hz")

    chunk_samples = int(chunk_duration * fs)
    total_samples = rec.get_num_frames()
    output_file = eeg_folder / 'eeg_data.dat'
    
    with open(output_file, 'wb') as f:
        start = 0
        while start < total_samples:
            end = min(start + chunk_samples, total_samples)
            chunk_data = rec.get_traces(start_frame=start, end_frame=end)
            
            # Pick every Nth sample
            decimated = chunk_data[::decimation_factor, :].astype('int16')
            decimated.tofile(f)
            
            start = end
            # logger.debug(f"Processed {end}/{total_samples} samples")
    logger.success(f"EEG data saved: {output_file}")

In [ ]:
rec_paths = [Path(p).resolve() for p in test_paths]
probe_filter = p['probe_filter']
probe_recs = {prb: [] for prb in probe_filter}
local_output = p['local_output']

# Check if concatenated probe data exists
for probe_idx, probe in enumerate(probe_recs.keys(), 1):
    probe_path = local_output / probe / 'concat'
    bin_path = probe_path / 'traces_cached_seg0.raw'
    if bin_path.exists():
        logger.info(f"Found concatenated data at {bin_path}, skipping concatenation.")
        saved_rec = si.load_extractor(bin_path)
        probe_recs[probe] = saved_rec
        log_recording(saved_rec)
        continue

# Load recordings for each session, group them by probe
for session_idx, session_path in enumerate(rec_paths, 1):
    logger.info("Loading probes.")
    logger.info(f"Session {session_idx}/{len(rec_paths)}: {session_path.name}")
    
    # Discover probes and ADC streams
    stream_names, stream_ids = se.get_neo_streams('openephysbinary', session_path)
    for stream_name, stream_id in zip(stream_names, stream_ids):
        # Extract probe name (e.g., "OneBox-0.ProbeA" -> "ProbeA")
        probe = stream_name.split(".")[-1]

        # Skip if: SYNC channel, not in filter, or already loaded
        if "SYNC" in stream_name or probe not in probe_filter or probe_recs[probe]:
            continue

        # Load recordings as OpenEphysBinaryExtractor objects
        rec = se.read_openephys(session_path, stream_id=stream_id)
        probe_recs[probe] = rec
    
    logger.success("Probes loaded.")

probe_concat = {}
# Concatenate, downsample, and save
for probe_idx, (probe, recs) in enumerate(probe_recs.items(), 1):
    logger.info(f"[{probe_idx}/{len(probe_recs)}]")

    logger.info(f"Concatenating {len(recs)} session(s)...")
    concat_rec = si.concatenate_recordings(recs) if len(recs) > 1 else recs[0]
    log_recording(concat_rec)
    logger.info(f"Sessions concatenated: {len(recs)}")
    
    
    probe_dir = local_output / probe
    concat_dir = probe_dir / 'concat'
    concat_dir.mkdir(parents=True, exist_ok=True)

    logger.info(f"Saving to: {concat_dir}")
    saved_rec = concat_rec.save(
        folder=concat_dir,
        **save_kwargs
    )
    logger.success(f"Saved concatenated recording")

    if probe == 'OneBox-ADC':
        continue  # Skip EEG downsampling for ADC

    probe_concat[probe] = saved_rec
    if target_fs:
        downsample_eeg(probe_dir, rec=saved_rec, target_fs=target_fs)
    
    

2025-11-04 12:46:08 | INFO     | Session 1/2: AA001_2025-10-16_14-01-10_1probe_test
2025-11-04 12:46:08 | INFO     | Session 2/2: 005_2025-10-16_16-42-37_1probe_test


In [65]:
dict = {'ProbeA': []}
if dict['ProbeA']:
    print('E')
dict['ProbeA'] = np.array(100)
if dict['ProbeA']:
    print('Y')

Y


In [61]:
dict

{'ProbeA': array(100)}

In [ ]:

def concat(probe_recordings, output_path, save_kwargs, target_fs=None):
    """Concatenate recordings across sessions and save as binary. Optional EEG downsampling."""
    logger.info("CONCATENATING RECORDINGS")
    probe_concat = {}
    
    for probe_idx, (probe_name, rec_list) in enumerate(probe_recordings.items(), 1):
        logger.info(f"[{probe_idx}/{len(probe_recordings)}] Processing {probe_name}")
        
        try:
            probe_folder = output_path / probe_name
            concat_folder = probe_folder / "concat"
            
            # Check if concatenation already exists
            binary_file = concat_folder / 'traces_cached_seg0.raw'
            if binary_file.exists():
                logger.info(f"Concatenation already exists, loading from disk...")

                saved_rec = si.load_extractor(concat_folder)
                log_recording(saved_rec)

                logger.success(f"Loaded existing concatenated recording")
            else:
                # Perform concatenation across sessions
                logger.info(f"Concatenating {len(rec_list)} session(s)...")
                concat_rec = si.concatenate_recordings(rec_list) if len(rec_list) > 1 else rec_list[0]
                log_recording(concat_rec)
                logger.info(f"Sessions concatenated: {len(rec_list)}")
                
                concat_folder.mkdir(parents=True, exist_ok=True)

                logger.info(f"Saving to: {concat_folder}")
                saved_rec = concat_rec.save(
                    folder=concat_folder,
                    **save_kwargs
                )
                logger.success(f"Saved concatenated recording")
            
            if "ADC" in probe_name:
                continue

            # EEG downsampling
            if target_fs:
                downsample_eeg(probe_folder, rec=saved_rec, target_fs=target_fs)

            probe_concat[probe_name] = saved_rec

        except Exception as e:
            logger.error(f"Failed to concatenate {probe_name}: {e}")
            logger.exception("Full traceback:")
            continue

    logger.success(f"Successfully processed {len(probe_concat)}/{len(probe_recordings)} probe(s)")
    return probe_concat